# 🧪 Praktikum 12 – Capstone Projekt & Semester-Review
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Alle Bausteine (Pipeline, Security, Eval) in einer Case-Study vereinen · Den State-of-the-Art reflektieren · Vorbereitung der finalen Abgabe

⏱️ **Dauer:** 90 Minuten

In [3]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is required for package installation. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
model_aliases = {MODEL, f"{MODEL}:latest"}
if not any(name in model_aliases for name in available_models):
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))
print("Systems ready for the capstone session.")

pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠏ 

Runtime: Local/Jupyter
Model: qwen3.5:0.8b
Available local models: qwen3.5:0.8b, code-reviewer:latest, nomic-embed-text:latest, devstral-small-2:latest, devstral-small-2:24b
Systems ready for the capstone session.


pulling manifest ⠏ pulling manifest 
pulling afb707b6b8fa: 100% ▕██████████████████▏ 1.0 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling b14c6eab49f9: 100% ▕██████████████████▏  476 B                         
verifying sha256 digest 
writing manifest 
success 


## Teil 1 – Case Study: Der 'Legal Assistant' ⏱️ 50 min
Wir bauen ein System, das Gesetzestexte (Kontext) nutzt, um Fragen schrittweise zu beantworten, dabei PII maskiert und seine Antwort selbst validiert.

In [8]:
class LegalBot:
    def __init__(self, model):
        self.model = model
        self.law_context = "§1: Jeder hat das Recht auf freie Entfaltung seiner Persönlichkeit. §2: Das Leben und die körperliche Unversehrtheit sind unverletzlich."
        self.options = {"temperature": 0, "num_predict": 80}

    def _ask(self, prompt, stop=None):
        response = ollama.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            think=False,
            options={**self.options, **({"stop": stop} if stop else {})},
            keep_alive="10m",
        )["message"]["content"].strip()
        if not response:
            raise RuntimeError("The model returned an empty response.")
        return response

    def run(self, user_query):
        print(f"--- Phase 1: PII Masking ---")
        safe_query = re.sub(r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', '[NAME]', user_query)

        print(f"--- Phase 2: Reasoning (CoT) ---")
        prompt = (
            f"Kontext: {self.law_context}\n"
            f"Anfrage: {safe_query}\n"
            "Antworte nur mit genau 2 kurzen Saetzen auf Deutsch. "
            "Nutze ausschliesslich den gegebenen Kontext. "
            "Triff keine Annahmen ueber die Person. "
            "Die Antwort soll sinngemaess diesem Muster folgen: "
            "'Nicht uneingeschraenkt: §1 schuetzt die freie Entfaltung der Persoenlichkeit. "
            "§2 setzt Grenzen, weil Leben und koerperliche Unversehrtheit unverletzlich sind.'"
        )
        res = self._ask(prompt, stop=["\n\n"])

        print(f"--- Phase 3: Self-Validation ---")
        check = (
            "Pruefe nur diese zwei Fakten aus dem Kontext:\n"
            "1. §1 schuetzt die freie Entfaltung der Persoenlichkeit.\n"
            "2. §2 schuetzt Leben und koerperliche Unversehrtheit und setzt damit Grenzen.\n"
            f"Antwort: {res}\n"
            "Wenn die Antwort nur diese Fakten wiedergibt oder paraphrasiert, antworte exakt mit JA. "
            "Wenn irgendeine zusaetzliche Behauptung enthalten ist, antworte exakt mit NEIN."
        )
        valid = self._ask(check, stop=["\n"])
        if valid not in {"JA", "NEIN"}:
            raise RuntimeError(f"Unexpected validation label: {valid}")

        return res, valid

bot = LegalBot(MODEL)
answer, is_valid = bot.run("Ich bin Max Mustermann. Darf ich tun was ich will?")
print(f"\nAntwort: {answer}\n\nValidität: {is_valid}")

--- Phase 1: PII Masking ---
--- Phase 2: Reasoning (CoT) ---
--- Phase 3: Self-Validation ---

Antwort: Nicht uneingeschraenkt: §1 schuetzt die freie Entfaltung der Persoenlichkeit. §2 setzt Grenzen, weil Leben und koerperliche Unversehrtheit unverletzlich sind.

Validität: JA


## Teil 2 – Architektur-Reflexion ⏱️ 20 min
Skizziere (mental oder auf Papier) den Datenfluss Ihres Projekts.

**Fragen:**
- Wo entstehen die höchsten Latenzen?
- An welcher Stelle ist das Risiko für 'Information Leakage' am größten?
- Welche Komponente würden Sie austauschen, wenn unbegrenzte GPU-Ressourcen verfügbar wären?

## 🏁 Der Weg zum Professional Deployment ⏱️ 20 min
Diskussion über: Docker-Containerisierung von Ollama, Monitoring von Halluzinationen in Produktion und Ethical AI Guidelines.

**Ende des Praktikums.**